# ClarifySQL · Notebook 08 — Fictional university demo
Use this after the final evaluation. The setup cell obtains the updated code from GitHub.
Attach the saved output of `sathviktr/phase6` run 363373340. One T4 is enough. Enable Internet.
With the default settings, **Save & Run All** finishes without asking for input and saves
`/kaggle/working/checkpoint8_results.zip`. Return that ZIP for analysis. No training is needed.

The seven automatic checks use fixed replies and made-up records. They test model behavior,
including changed ranking preferences and age cutoffs; they are separate from the benchmark.
Every record is made up. The model's question, explanation and SQL are displayed as plain text.
The demo has no file/database upload and does not expose the sample-values tool.

In [ ]:
import os, sys, json, zipfile, pathlib, subprocess, time, hashlib, shutil
from pathlib import Path
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
assert WORK.exists(), 'Use Kaggle with a GPU for this notebook.'
REPO = WORK / 'LMA_ClarifySQL'
SOURCE = 'github'  # 'archive' is an optional source-ZIP fallback
REPO_URL = 'https://github.com/Sathviktr/LMA_ClarifySQL.git'
BRANCH = 'main'
REVISION = ''  # optional: exact commit printed by an earlier run, for reproduction
if SOURCE == 'github':
    if (REPO / '.git').exists():
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
        if not REVISION:
            subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', f'origin/{BRANCH}'], check=True)
    else:
        assert not REPO.exists() or not any(REPO.iterdir()), 'Use a fresh session or SOURCE=archive for a ZIP checkout.'
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
    if REVISION:
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REVISION], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REVISION], check=True)
    commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    source_info = {'mode': 'github', 'commit': commit, 'branch': BRANCH}
    print('GitHub source commit:', commit)
elif SOURCE == 'archive':
    sources = list(INPUT.rglob('clarifysql_checkpoint7_source.zip'))
    REPO.mkdir(exist_ok=True)
    if len(sources) == 1:
        with zipfile.ZipFile(sources[0]) as z:
            for member in z.infolist():
                dest = (REPO / member.filename).resolve()
                assert dest.is_relative_to(REPO.resolve()), 'Unsafe source archive path'
            z.extractall(REPO)
    else:
        # Kaggle may automatically unpack an uploaded source ZIP into the dataset.
        manifests = list(INPUT.rglob('CHECKPOINT7_SOURCE.json'))
        assert not sources and len(manifests) == 1, 'Attach exactly one checkpoint-7 source dataset.'
        source_root = manifests[0].parent.resolve()
        source_files = json.loads(manifests[0].read_text())['files']
        for relative in [*source_files, 'CHECKPOINT7_SOURCE.json']:
            src, dest = (source_root / relative).resolve(), (REPO / relative).resolve()
            assert src.is_relative_to(source_root) and dest.is_relative_to(REPO.resolve()), 'Unsafe source path'
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest)
    source_manifest = json.loads((REPO / 'CHECKPOINT7_SOURCE.json').read_text())
    for relative, expected in source_manifest['files'].items():
        assert hashlib.sha256((REPO / relative).read_bytes()).hexdigest() == expected, relative
    source_info = {'mode': 'archive', 'manifest_sha256': hashlib.sha256(
        (REPO / 'CHECKPOINT7_SOURCE.json').read_bytes()).hexdigest()}
    print('Source files verified:', len(source_manifest['files']))
else:
    raise ValueError('SOURCE must be github or archive')
required = ['scripts/run_final.py', 'src/clarifysql/eval/final_run.py',
            'requirements-eval.txt', 'data/episodes/slot_terms.json']
missing = [p for p in required if not (REPO / p).is_file()]
assert not missing, f'Evaluation code missing: {missing}. Push the new files to GitHub before running.'
sys.path.insert(0, str(REPO / 'src'))
os.environ['CLARIFYSQL_DATA'] = '/tmp/clarifysql_data'
def checked(args, **kwargs):
    return subprocess.run(args, cwd=REPO, check=True, **kwargs)
checked([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-eval.txt'])
checked([sys.executable, '-m', 'pip', 'install', '-q', 'pytest', 'nbformat', 'numpy'])
checked([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'])
# The old preinstalled torchao can prevent PEFT from loading these adapters.
checked([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])

## 1. Load the selected adapter
Keep `RUN_INTERACTIVE = False` for the first saved run. If Kaggle offers T4 x2, that also works;
the model uses the first GPU. The attached output must contain the weights, not just the result ZIP.

In [ ]:
RUN_INTERACTIVE = False  # enable later in an interactive session to try the web interface
assert (REPO / 'src/clarifysql/demo/checks.py').is_file(), 'Update the repository before running notebook 08.'
checked([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-demo.txt'])
checked([sys.executable, '-m', 'pytest', 'tests/test_demo.py', 'tests/test_demo_checks.py',
         '-p', 'no:cacheprovider', '-q'])
import torch
assert torch.cuda.is_available(), 'Enable a T4 GPU.'
roots = [p.parent for p in INPUT.rglob('choice.json')
         if (p.parent / 'sft_s1' / 'adapter_model.safetensors').is_file()
         and (p.parent / 'sft_s1' / 'adapter_config.json').is_file()]
assert len(roots) == 1, 'Attach exactly one completed phase6 notebook output.'
ADAPTER = roots[0] / 'sft_s1'
choice = json.loads((roots[0] / 'choice.json').read_text())
assert choice['lr'] == '2e-4' and choice['epoch'] == 3, 'Unexpected adapter selection.'
from clarifysql.demo.university import create_database
from clarifysql.llm.backend import HFBackend
from clarifysql.eval.artifacts import write_json
DEMO = WORK / 'checkpoint8_demo'
DEMO.mkdir(exist_ok=True)
db = create_database(DEMO / 'university.sqlite')
model = HFBackend(adapter_path=str(ADAPTER))
write_json(DEMO / 'source.json', {'source': source_info, 'choice': choice,
    'adapter_source': 'sathviktr/phase6 run 363373340', 'interactive_requested': RUN_INTERACTIVE})
print('Loaded sft_s1 on:', torch.cuda.get_device_name(0))

## 2. Run seven checks and save every outcome
Clear requests should execute, the glossary should handle high marks, and missing email data should
cause abstention. The ambiguous examples use predefined replies. The age and ranking variants check
that a reply can change the answer. No simulated-user model is called here.

A failed check is saved and does not prevent packaging. The rows, SQL, questions, replies, errors,
model revision and source hashes are kept. A matching rerun resumes; changed inputs require a new
`DEMO` directory. Extra result columns count as a mismatch under the strict reference check.

In [ ]:
from clarifysql.demo.checks import run_demo_checks
summary = run_demo_checks(model, db, DEMO / 'checks', adapter_path=ADAPTER)
assert summary['backend'] == 'hf' and summary['complete'], 'Incomplete or mock results are not a completed GPU demo run.'
print(f"Completed {summary['cases']} cases: {summary['passed']} passed.")
print('Failed cases:', summary['failed'])

## 3. Check interface construction and package the output
Building the Gradio interface is recorded separately from a real browser interaction. If construction
fails, its error is included in the ZIP. The automatic check does not claim a live user session occurred.
Download **checkpoint8_results.zip** from this saved version's output. Model weights are not included.

In [ ]:
from clarifysql.demo.app import create_app, launch
app = None
try:
    app = create_app(model, db)
    interface = {'constructed': True, 'browser_interaction_checked': False, 'error': None}
except Exception as exc:
    interface = {'constructed': False, 'browser_interaction_checked': False,
                 'error': f'{type(exc).__name__}: {exc}'}
write_json(DEMO / 'interface.json', interface)
ZIP = WORK / 'checkpoint8_results.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(DEMO.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(DEMO).as_posix())
print('Download:', ZIP, 'bytes:', ZIP.stat().st_size)
print('Interface:', interface)
from IPython.display import FileLink, display
display(FileLink(str(ZIP)))

## 4. Optional: try your own replies in the web interface
For the first saved run this cell skips interaction, so the ZIP can be downloaded normally.
To try the interface later, set `RUN_INTERACTIVE = True` and run this cell interactively.
Enter a username and a password when prompted, then open the temporary authenticated share link.
These credentials are not saved in the result ZIP. Stop the notebook when finished.

In [ ]:
if RUN_INTERACTIVE:
    assert app is not None, 'Interface construction failed; see interface.json in the result ZIP.'
    from getpass import getpass
    os.environ['CLARIFYSQL_DEMO_USER'] = input('Demo username: ').strip()
    os.environ['CLARIFYSQL_DEMO_PASSWORD'] = getpass('Demo password: ')
    launch(app, share=True)
else:
    print('Automatic run complete. Return checkpoint8_results.zip; no interactive input is needed.')